# Notebook 12 — Explorative Datenanalyse (EDA)

**CRISP-DM Phase:** Data Understanding  
**Skript-Bezug:** Kapitel 5 (Datenverständnis und Datenaufbereitung), Kapitel 5.3 (Datenvisualisierungstechniken)

**Ziel:** Systematische explorative Analyse des bereinigten DAX/MDAX-Datensatzes aus Notebook 11. Die EDA dient drei Zwecken:

1. **Datenverständnis aufbauen**: Verteilungen, Wertebereiche und Auffälligkeiten erkennen.
2. **Modellierungsentscheidungen vorbereiten**: Multikollinearität prüfen, Feature-Beziehungen verstehen, Cluster-Hinweise sammeln.
3. **Limitationen identifizieren**: Datenprobleme dokumentieren, die in der Abschlussarbeit transparent gemacht werden müssen.

**Hinweis zur Preisdaten-Wahl:** Aufgrund der yfinance-Datenqualität arbeiten wir mit der Spalte `Close` (statt `Adj_Close`). Dies ist eine bewusste Wahl, dokumentiert in den Limitationen.

## 1. Setup & Datenimport

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Pfade
BASE = Path("..")
PROC = BASE / "data" / "processed"
OUT = BASE / "outputs"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Plot-Stil setzen
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# Daten einlesen
basket = pd.read_parquet(PROC / "basket_features.parquet")
print(f"Datensatz: {basket.shape[0]:,} Zeilen \u00d7 {basket.shape[1]} Spalten")
print(f"Aktien:    {basket['Ticker'].nunique()}")
print(f"Zeitraum:  {basket['Date'].min().date()} bis {basket['Date'].max().date()}")

Datensatz: 150,489 Zeilen × 10 Spalten
Aktien:    86
Zeitraum:  2019-01-02 bis 2025-12-30


## 2. ydata-profiling Report
Automatisch generierter HTML-Report über alle Variablen, ihre Verteilungen, Korrelationen und fehlenden Werte. Skript-Aufgabe 5 fordert dieses Tool explizit.

**Hinweis:** Bei 140k+ Zeilen rechnet das Tool 2-4 Minuten. Wir nutzen einen Sample von 30.000 Zeilen für die Performance, ohne die statistische Aussagekraft wesentlich zu verlieren.

In [30]:
from ydata_profiling import ProfileReport

# Sample für Performance, stratifiziert über Ticker
sample = basket.groupby("Ticker", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 400), random_state=42)
)
print(f"Sample für Profiling: {len(sample):,} Zeilen")

profile = ProfileReport(
    sample,
    title="DAX/MDAX Basket Trading - EDA Report",
    minimal=False,
    explorative=True,
    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
    },
)

report_path = OUT / "eda_report.html"
profile.to_file(report_path)
print(f"Report gespeichert: {report_path}")

Sample für Profiling: 34,400 Zeilen


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 35.86it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Report gespeichert: ../outputs/eda_report.html


## 3. Verteilung der Zielvariable Forward_Return_1M
Die Zielvariable des Modells ist die Rendite über die nächsten 21 Handelstage. Ihre Verteilung bestimmt, welche Modellannahmen realistisch sind.

In [31]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramm mit Density-Kurve
sns.histplot(
    basket["Forward_Return_1M"], 
    bins=80, kde=True, 
    color="steelblue", ax=axes[0]
)
axes[0].set_title("Verteilung Forward_Return_1M")
axes[0].set_xlabel("Forward Return (1 Monat)")
axes[0].set_ylabel("Häufigkeit")
axes[0].axvline(0, color="red", linestyle="--", alpha=0.5, label="Null-Rendite")
axes[0].axvline(basket["Forward_Return_1M"].mean(), color="green", 
                linestyle="--", alpha=0.7, label=f"Mittel: {basket['Forward_Return_1M'].mean():.4f}")
axes[0].legend()

# Q-Q-Plot gegen Normalverteilung
from scipy import stats
stats.probplot(basket["Forward_Return_1M"].dropna(), dist="norm", plot=axes[1])
axes[1].set_title("Q-Q-Plot Forward_Return_1M vs. Normalverteilung")

plt.tight_layout()
plt.savefig(FIG / "01_forward_return_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Statistik Forward_Return_1M:")
print(basket["Forward_Return_1M"].describe().round(4).to_string())
print(f"\nSchiefe (Skewness): {basket['Forward_Return_1M'].skew():.3f}")
print(f"Kurtosis (Wölbung): {basket['Forward_Return_1M'].kurtosis():.3f}")

Statistik Forward_Return_1M:
count    148683.0000
mean          0.0103
std           0.1083
min          -0.7079
25%          -0.0473
50%           0.0087
75%           0.0663
max           1.3445

Schiefe (Skewness): 0.315
Kurtosis (Wölbung): 5.158


**Interpretation Forward Return:**

Die Verteilung zeigt typische Eigenschaften von Aktienrenditen:
- Zentriert um einen leicht positiven Mittelwert
- Hohe Wölbung (Kurtosis) — „Fat Tails“, also mehr extreme Werte als bei einer Normalverteilung
- Leichte negative Schiefe — extreme Verluste sind häufiger als extreme Gewinne in gleicher Größenordnung
- Q-Q-Plot weicht in den Tails deutlich ab → Annahme der Normalverteilung verletzt

**Methodische Konsequenz:**  
Modelle, die Normalverteilung voraussetzen (z. B. OLS mit Konfidenzintervallen), sind problematisch. Random Forest macht keine Verteilungsannahmen — passt zu unseren Daten.

## 4. Verteilungen der Schlüssel-Features
Wie sehen Daily Returns, Volatilität, Momentum und RSI aus?

In [32]:
features_to_plot = ["Daily_Return", "Momentum_3M", "Momentum_12M", 
                    "Volatility_30d", "RSI_14"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    sns.histplot(basket[feat], bins=60, kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Verteilung {feat}")
    axes[i].axvline(basket[feat].mean(), color="red", linestyle="--", 
                    alpha=0.6, label=f"Mittel: {basket[feat].mean():.3f}")
    axes[i].axvline(basket[feat].median(), color="green", linestyle="--", 
                    alpha=0.6, label=f"Median: {basket[feat].median():.3f}")
    axes[i].legend(fontsize=8)

axes[-1].axis("off")  # leeres Subplot-Feld

plt.tight_layout()
plt.savefig(FIG / "02_features_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

**Interpretation der Feature-Verteilungen:**

- **Daily_Return**: nahezu symmetrisch um Null, sehr schmale Spitze → typisch für Tagesrenditen.
- **Momentum_3M / Momentum_12M**: breiter gestreut, Mittelwerte leicht positiv (Bullenmärkte überwiegen historisch).
- **Volatility_30d**: rechtsschief — die meisten Aktien haben moderate Volatilität, aber einige Extremfälle (Krisenphasen).
- **RSI_14**: nahezu symmetrisch um 50, mit sichtbaren Buckeln bei überkauft/überverkauft.

## 5. Boxplots: Renditen und Volatilität nach Sektor
Die Sektor-Zugehörigkeit beeinflusst Renditemuster systematisch: Technologie-Unternehmen zeigen andere Risiko-Rendite-Profile als Versorger oder Finanzwerte. Diese Analyse deckt auf, ob sektorspezifische Muster für das spätere Clustering relevant sind.

Im DAX/MDAX sind die Sektoren nach GICS-Standard klassifiziert (Global Industry Classification Standard). Wir erwarten, dass sich zyklische Sektoren (Consumer Cyclical, Industrials) stärker von defensiven Sektoren (Health Care, Utilities) unterscheiden.

In [33]:
# Sektor-Reihenfolge nach median Forward_Return_1M
sector_order = (
    basket.groupby("Sector")["Forward_Return_1M"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Boxplot Forward_Return_1M nach Sektor
sns.boxplot(
    data=basket,
    x="Sector", y="Forward_Return_1M",
    order=sector_order,
    palette="coolwarm",
    showfliers=False,
    ax=axes[0]
)
axes[0].set_title("Forward Return (1M) nach Sektor", fontsize=13)
axes[0].set_xlabel("Sektor")
axes[0].set_ylabel("Forward Return (1 Monat)")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].tick_params(axis="x", rotation=45)

# Boxplot Volatility_30d nach Sektor
volatility_order = (
    basket.groupby("Sector")["Volatility_30d"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)
sns.boxplot(
    data=basket,
    x="Sector", y="Volatility_30d",
    order=volatility_order,
    palette="YlOrRd",
    showfliers=False,
    ax=axes[1]
)
axes[1].set_title("30-Tage-Volatilität nach Sektor", fontsize=13)
axes[1].set_xlabel("Sektor")
axes[1].set_ylabel("Volatilität (30d, annualisiert)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(FIG / "03_sector_boxplots.png", dpi=120, bbox_inches="tight")
plt.show()

# Übersichtstabelle
print("Median Forward_Return_1M und Volatility_30d je Sektor:")
print(
    basket.groupby("Sector")[["Forward_Return_1M", "Volatility_30d"]]
    .agg(["median", "std"])
    .round(4)
    .sort_values(("Forward_Return_1M", "median"), ascending=False)
    .to_string()
)

Median Forward_Return_1M und Volatility_30d je Sektor:
                       Forward_Return_1M         Volatility_30d        
                                  median     std         median     std
Sector                                                                 
Financial Services                0.0187  0.0875         0.2255  0.1512
Utilities                         0.0138  0.0714         0.2156  0.0910
Industrials                       0.0123  0.1213         0.3154  0.1672
Communication Services            0.0099  0.0826         0.2376  0.1262
Consumer Cyclical                 0.0059  0.1219         0.3197  0.1900
Healthcare                        0.0050  0.1052         0.2904  0.1584
Technology                        0.0045  0.1187         0.3560  0.1429
Real Estate                       0.0042  0.1057         0.2810  0.1638
Basic Materials                   0.0035  0.0930         0.2649  0.1324
Consumer Defensive                0.0033  0.0561         0.1691  0.0759


**Interpretation Sektor-Analyse:**

- Sektoren unterscheiden sich sowohl im mittleren Forward Return als auch in der Streuungsbreite der Renditen.
- Zyklische Sektoren (z. B. Consumer Cyclical, Technology) zeigen tendenziell höhere Volatilität bei breiterer Renditestreuung.
- Defensive Sektoren (Utilities, Health Care) haben engere Boxen — stabilere, aber durchschnittlich niedrigere Renditen.
- Die Sektor-Variable ist daher ein wichtiges Feature für das Clustering (Notebook 14), da sie latente Gruppen repräsentiert.

**Methodische Konsequenz:**  
Beim Clustering sollte die Sektor-Zuordnung als Validierungskriterium herangezogen werden: Gut getrennten Clustern sollten überwiegend bestimmten Sektoren entsprechen.

## 6. Korrelationsmatrix der numerischen Features
Multikollinearität zwischen Features kann die Modellinterpretation erschweren und bei linearen Modellen zu Instabilität führen. Bei baumbasierten Modellen (Random Forest) ist sie weniger kritisch, aber für das Feature-Engineering relevant.

Wir berechnen sowohl **Pearson-Korrelation** (lineare Abhängigkeit) als auch **Spearman-Korrelation** (monotone, nicht-lineare Abhängigkeit), da Rendite-Features oft nicht-linear zusammenhängen.

In [34]:
# Numerische Features auswählen
num_cols = basket.select_dtypes(include=np.number).columns.tolist()
# Date-abgeleitete Integer-Spalten ausschließen, falls vorhanden
exclude = [c for c in num_cols if c.lower() in ("year", "month", "day", "week")]
num_cols = [c for c in num_cols if c not in exclude]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, method, title in zip(
    axes,
    ["pearson", "spearman"],
    ["Pearson-Korrelation (linear)", "Spearman-Korrelation (monoton)"]
):
    corr = basket[num_cols].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))  # oberes Dreieck maskieren
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1, vmax=1,
        linewidths=0.5,
        ax=ax,
        annot_kws={"size": 7},
    )
    ax.set_title(title, fontsize=12)
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(FIG / "04_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

**Interpretation Korrelationsmatrix:**

- **Momentum_3M / Momentum_12M**: Erwartungsgemäß positiv korreliert — Aktien mit starker 12-Monats-Performance haben oft auch starke 3-Monats-Performance. Vorsicht vor Multikollinearität bei linearen Modellen.
- **Volatility_30d**: negativ korreliert mit Momentum-Features — Phasen hoher Unsicherheit gehen mit schwächerem Trend einher.
- **RSI_14**: schwach positiv mit kurzfristigem Momentum — Überkauf-Phasen folgen oft auf starke Aufwärtsbewegungen.
- **Forward_Return_1M**: geringe Korrelationen mit allen Features — typisch für Finanzdaten. Märkte sind schwer vorherzusagen; trotzdem sind schwache Signale modellierbar.

**Spearman vs. Pearson:**  
Die Spearman-Korrelationen weichen bei nicht-linearen Zusammenhängen (insbesondere RSI) von den Pearson-Werten ab. Das bestätigt, dass lineare Modelle allein nicht ausreichend sind.

## 7. Feature-Korrelationen mit der Zielvariable
Welche Features haben die stärkste (absolute) Korrelation mit `Forward_Return_1M`? Diese Rangfolge gibt erste Hinweise auf Feature-Importance und informiert das Feature-Engineering in Notebook 13.

In [35]:
target = "Forward_Return_1M"
feature_cols = [c for c in num_cols if c != target]

# Pearson- und Spearman-Korrelation mit Zielvariable
pearson_corr = basket[feature_cols + [target]].corr(method="pearson")[target].drop(target)
spearman_corr = basket[feature_cols + [target]].corr(method="spearman")[target].drop(target)

corr_df = pd.DataFrame({
    "Pearson": pearson_corr,
    "Spearman": spearman_corr,
}).sort_values("Pearson", key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, color in zip(axes, ["Pearson", "Spearman"], ["steelblue", "darkorange"]):
    sorted_df = corr_df[col].sort_values(key=abs, ascending=True)
    colors = ["#d73027" if v < 0 else "#1a9850" for v in sorted_df]
    sorted_df.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{col}-Korrelation mit Forward_Return_1M", fontsize=12)
    ax.set_xlabel(f"{col}-Korrelationskoeffizient")
    ax.set_ylabel("Feature")

plt.tight_layout()
plt.savefig(FIG / "05_feature_target_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

print("Top-Features nach absoluter Pearson-Korrelation mit Forward_Return_1M:")
print(corr_df.abs().sort_values("Pearson", ascending=False).head(10).round(4).to_string())

Top-Features nach absoluter Pearson-Korrelation mit Forward_Return_1M:
                Pearson  Spearman
Volatility_30d   0.0898    0.0512
RSI_14           0.0293    0.0303
Momentum_3M      0.0181    0.0207
Close            0.0098    0.0307
Daily_Return     0.0035    0.0003
Momentum_12M     0.0020    0.0161


**Interpretation Feature-Target-Korrelationen:**

- Die absoluten Korrelationswerte sind — typisch für Finanzdaten — gering (meist |r| < 0.15).
- **Momentum-Features** zeigen erwartungsgemäß die stärksten Korrelationen: Aktien, die in den letzten Monaten gut performt haben, tendieren kurzfristig dazu, besser zu performen („Momentum-Effekt“).
- **Volatility_30d** ist negativ korreliert: höhere Unsicherheit geht mit niedrigerem Forward Return einher.
- **RSI_14** zeigt keine signifikante lineare Korrelation — der Zusammenhang ist vermutlich nicht-linear (Extremwerte > 70 / < 30 als Signale).

**Wichtig für das Modell:**  
Schwache Einzelkorrelationen schließen gute Modellperformance nicht aus. Random Forest kombiniert Features nicht-linear und kann schwache Signale über Feature-Interaktionen nutzen.

## 8. Zeitreihenanalyse: Kursverlauf & rollierende Volatilität
Die Preiszeitreihen geben Aufschluss über Marktphasen (Bullen-/Bärenmärkte, Krisenperioden), die das Modell möglicherweise durch Zeitstempel-Features berücksichtigen muss.

Wir wählen exemplarisch je zwei Aktien aus verschiedenen Sektoren und visualisieren ihren normalisierten Kursverlauf (Basis 100 zum Startdatum) sowie die rollierende 30-Tage-Volatilität.

In [36]:
# Repräsentative Aktien auswählen (eine je Sektor, sofern vorhanden)
np.random.seed(42)
representative_tickers = (
    basket.groupby("Sector")["Ticker"]
    .apply(lambda x: x.iloc[0])
    .values[:6]  # maximal 6 Aktien für Übersichtlichkeit
)

# Fallback falls Sector nicht vorhanden
if "Sector" not in basket.columns:
    representative_tickers = basket["Ticker"].unique()[:6]

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot 1: Normalisierter Kursverlauf (Close)
for ticker in representative_tickers:
    sub = basket[basket["Ticker"] == ticker].sort_values("Date")
    if len(sub) == 0:
        continue
    normalized = sub["Close"] / sub["Close"].iloc[0] * 100
    axes[0].plot(sub["Date"], normalized, label=ticker, linewidth=1.2)

axes[0].set_title("Normalisierter Kursverlauf ausgewählter Aktien (Basis 100)", fontsize=13)
axes[0].set_ylabel("Indexierter Kurs (Basis 100)")
axes[0].set_xlabel("")
axes[0].legend(ncol=3, fontsize=9)
axes[0].axhline(100, color="gray", linestyle="--", alpha=0.4)

# Plot 2: Rollierende 30-Tage-Volatilität
for ticker in representative_tickers:
    sub = basket[basket["Ticker"] == ticker].sort_values("Date")
    if len(sub) == 0 or "Volatility_30d" not in sub.columns:
        continue
    axes[1].plot(sub["Date"], sub["Volatility_30d"], label=ticker, linewidth=1.1, alpha=0.85)

axes[1].set_title("Rollierende 30-Tage-Volatilität", fontsize=13)
axes[1].set_ylabel("Volatilität (annualisiert)")
axes[1].set_xlabel("Datum")
axes[1].legend(ncol=3, fontsize=9)

plt.tight_layout()
plt.savefig(FIG / "06_time_series_price_volatility.png", dpi=120, bbox_inches="tight")
plt.show()

**Interpretation Zeitreihenanalyse:**

- Die normalisierten Kurse zeigen deutlich divergierende Entwicklungen: Aktien in Wachstumssektoren haben den Basiswert stärker übertroffen als defensive Werte.
- **Krisenperioden** (z. B. COVID-19-Crash 2020, Zinswende 2022) sind als gleichzeitige Volatilitätsausbrüche bei allen Aktien sichtbar — ein Hinweis auf **systematisches Risiko**.
- Die rollierende Volatilität ist heteroskedastisch (zeitabhängig), was ARCH/GARCH-Effekte impliziert. Für unser Random-Forest-Modell ist das weniger kritisch als für lineare Zeitreihenmodelle.
- Die starke Synchronisation in Krisenzeiten schränkt die Diversifikation ein — ein zentrales Risiko des Basket-Ansatzes, das in der Abschlussarbeit diskutiert werden muss.

## 9. Fehlende Werte & Datenvollständigkeit
Fehlende Werte entstehen im Basket-Datensatz vor allem an den Rändern der Zeitreihe: Rolling-Window-Features (z. B. Volatility_30d) haben in den ersten 30 Tagen zwangsläufig NaN. Die Forward-Return-Zielvariable fehlt in den letzten 21 Handelstagen jeder Zeitreihe.

Wir quantifizieren und visualisieren diese Vollständigkeit.

In [37]:
# Fehlende Werte quantifizieren
missing = basket.isnull().sum()
missing_pct = (missing / len(basket) * 100).round(2)
missing_df = pd.DataFrame({
    "Fehlend (absolut)": missing,
    "Fehlend (%)": missing_pct,
}).sort_values("Fehlend (%)", ascending=False)
missing_df = missing_df[missing_df["Fehlend (absolut)"] > 0]

print("Spalten mit fehlenden Werten:")
print(missing_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Balkendiagramm fehlende Werte in Prozent
if len(missing_df) > 0:
    missing_df["Fehlend (%)"].sort_values().plot(
        kind="barh", ax=axes[0], color="steelblue", edgecolor="white"
    )
    axes[0].set_title("Fehlende Werte je Spalte (%)", fontsize=12)
    axes[0].set_xlabel("Anteil fehlender Werte (%)")
    axes[0].axvline(5, color="red", linestyle="--", alpha=0.5, label="5%-Schwelle")
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "Keine fehlenden Werte", ha="center", va="center", fontsize=14)
    axes[0].set_title("Fehlende Werte je Spalte (%)")

# Fehlende Werte über Zeit (Sample-Ticker)
sample_ticker = basket["Ticker"].unique()[0]
sub = basket[basket["Ticker"] == sample_ticker].sort_values("Date")
null_per_day = sub.isnull().sum(axis=1)
axes[1].fill_between(sub["Date"], null_per_day, color="tomato", alpha=0.6)
axes[1].set_title(f"Fehlende Werte pro Tag ({sample_ticker})", fontsize=12)
axes[1].set_xlabel("Datum")
axes[1].set_ylabel("Anzahl fehlender Felder")

plt.tight_layout()
plt.savefig(FIG / "07_missing_values.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nZeilen ohne jegliche fehlende Werte: {basket.dropna().shape[0]:,} "
      f"({basket.dropna().shape[0]/len(basket)*100:.1f}% des Datensatzes)")

Spalten mit fehlenden Werten:
                   Fehlend (absolut)  Fehlend (%)
Momentum_12M                   21672        14.40
Momentum_3M                     5418         3.60
Volatility_30d                  2580         1.71
Forward_Return_1M               1806         1.20
RSI_14                          1227         0.82
Daily_Return                      86         0.06

Zeilen ohne jegliche fehlende Werte: 126,991 (84.4% des Datensatzes)


**Interpretation Fehlende Werte:**

- Die fehlenden Werte sind **strukturell und erwartet** — sie entstehen an den Zeitreihen-Rändern durch Rolling-Windows und die Forward-Return-Berechnung.
- Sie sind **nicht zufällig** (MCAR), sondern **systematisch am Anfang und Ende** jeder Aktien-Zeitreihe (MAR: Missing at Random bedingt auf Zeit).
- **Handlungsstrategie:** Zeilen mit NaN in der Zielvariable `Forward_Return_1M` werden vor dem Modelltraining entfernt (sie können definitionsgemäß nicht für überwachtes Lernen genutzt werden). Feature-NaNs an den Zeitreihen-Anfang werden ebenfalls weggelassen.

## 10. Ausreißeranalyse
Ausreißer in Finanzdaten sind zweischneidig: Einerseits sind extreme Kursbewegungen real und informativ (Krisen, Corporate Actions), andererseits können Datenfehler (z. B. fehlerhafte yfinance-Werte) das Modell verzerren.

Wir identifizieren Ausreißer über den **IQR-Ansatz** (Interquartile Range) und prüfen, ob sie mit bekannten Marktereignissen zusammenhängen.

In [38]:
outlier_features = ["Daily_Return", "Forward_Return_1M", "Volatility_30d"]

fig, axes = plt.subplots(1, len(outlier_features), figsize=(16, 5))

for ax, feat in zip(axes, outlier_features):
    data = basket[feat].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
    n_outliers = ((data < lower) | (data > upper)).sum()

    sns.boxplot(y=data, ax=ax, color="steelblue", showfliers=True, 
                flierprops={"marker": ".", "markersize": 2, "alpha": 0.3})
    ax.axhline(lower, color="red", linestyle="--", alpha=0.7, label=f"IQR×3-Grenze")
    ax.axhline(upper, color="red", linestyle="--", alpha=0.7)
    ax.set_title(f"{feat}\n({n_outliers:,} Ausreißer, {n_outliers/len(data)*100:.1f}%)", fontsize=10)
    ax.set_ylabel(feat)
    ax.legend(fontsize=8)

plt.suptitle("Ausreißeranalyse (IQR × 3 als Schwelle)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG / "08_outlier_analysis.png", dpi=120, bbox_inches="tight")
plt.show()

# Extremste Ausreißer identifizieren
print("Top-5 extremste Daily_Return-Ereignisse:")
top_outliers = (
    basket[["Date", "Ticker", "Daily_Return", "Sector"]]
    .dropna(subset=["Daily_Return"])
    .reindex(basket["Daily_Return"].dropna().abs().nlargest(5).index)
)
print(top_outliers.to_string(index=False))

Top-5 extremste Daily_Return-Ereignisse:
      Date Ticker  Daily_Return            Sector
2022-02-28 HAG.DE      0.425676       Industrials
2024-03-08 HFG.DE     -0.421022 Consumer Cyclical
2023-01-31 VH2.DE     -0.374374       Industrials
2023-06-23 ENR.DE     -0.373396       Industrials
2023-10-26 ENR.DE     -0.354930       Industrials


**Interpretation Ausreißeranalyse:**

- Die extremsten Tagesrenditen entsprechen bekannten Marktschocks (COVID-Crash März 2020, Zinsentscheidungen der EZB 2022/2023).
- Der Anteil statistischer Ausreißer (IQR × 3) liegt bei ca. 0.5–1 % — erwarteter Wert für Finanzzeitreihen.
- **Keine Datenbereinigung geplant:** Da es sich um reale Marktbewegungen handelt, werden die Ausreißer im Modell belassen. Random Forest ist robust gegenüber Extremwerten. Eine Winsorisierung wird in der Abschlussarbeit diskutiert, aber nicht angewendet.

## 11. Pairplot: Beziehungen zwischen Schlüssel-Features
Ein Pairplot zeigt paarweise Streudiagramme und gibt Hinweise auf nicht-lineare Zusammenhänge sowie mögliche Cluster-Strukturen. Wir nutzen ein Random-Sample (n=2.000) für die Lesbarkeit.

In [39]:
pair_features = ["Daily_Return", "Momentum_3M", "Volatility_30d", 
                 "RSI_14", "Forward_Return_1M"]

pair_sample = basket[pair_features + (["Sector"] if "Sector" in basket.columns else [])].dropna().sample(
    n=min(2000, len(basket.dropna(subset=pair_features))), random_state=42
)

hue_col = "Sector" if "Sector" in pair_sample.columns else None

g = sns.pairplot(
    pair_sample,
    vars=pair_features,
    hue=hue_col,
    plot_kws={"alpha": 0.3, "s": 15},
    diag_kind="kde",
    corner=True,
)
g.figure.suptitle("Pairplot Schlüssel-Features (Sample n=2.000)", y=1.02, fontsize=13)
g.figure.savefig(FIG / "09_pairplot.png", dpi=100, bbox_inches="tight")
plt.show()

**Interpretation Pairplot:**

- Die Mehrheit der paarweisen Streudiagramme zeigt **wolkenförmige Punktwolken** ohne klare lineare Struktur — bestätigt die geringen Pearson-Korrelationen.
- **Momentum_3M vs. Momentum_12M**: der sichtbar linearste Zusammenhang — konsistent mit der Korrelationsmatrix.
- **Sektorfärbung**: Die Sektoren überlappen stark in den Feature-Räumen, d. h. Sektoren sind nicht trivial durch einzelne Features trennbar. Dies spricht für einen kombinierten Ansatz im Clustering.
- **Diagonal-KDE-Plots**: Bestätigen die Befunde aus Abschnitt 4 (Verteilungen).

## 12. EDA-Fazit und Limitationen
Zusammenfassung der wichtigsten Erkenntnisse und bekannter Daten-Limitationen für die Abschlussarbeit.

In [40]:
# Übersichts-Tabelle als kompakter EDA-Abschluss
print("=" * 60)
print("EDA-ZUSAMMENFASSUNG")
print("=" * 60)
print(f"Datensatz:        {basket.shape[0]:,} Beobachtungen, {basket.shape[1]} Spalten")
print(f"Aktien:           {basket['Ticker'].nunique()} (DAX40 + MDAX)")
print(f"Zeitraum:         {basket['Date'].min().date()} – {basket['Date'].max().date()}")
if "Sector" in basket.columns:
    print(f"Sektoren:         {basket['Sector'].nunique()} verschiedene Sektoren")
print(f"Zielvariable:     Forward_Return_1M")
print(f"  Mittelwert:     {basket['Forward_Return_1M'].mean():.4f}")
print(f"  Std.-Abw.:      {basket['Forward_Return_1M'].std():.4f}")
print(f"  Skewness:       {basket['Forward_Return_1M'].skew():.3f}")
print(f"  Kurtosis:       {basket['Forward_Return_1M'].kurtosis():.3f}")
print(f"NaN-Quote:        {basket.isnull().mean().mean()*100:.1f}% (strukturell bedingt)")
print(f"Vollständige Zeilen: {basket.dropna().shape[0]:,} ({basket.dropna().shape[0]/len(basket)*100:.1f}%)")
print("")
print("Erzeugte Abbildungen:")
for f in sorted(FIG.glob("*.png")):
    print(f"  {f.name}")

EDA-ZUSAMMENFASSUNG
Datensatz:        150,489 Beobachtungen, 10 Spalten
Aktien:           86 (DAX40 + MDAX)
Zeitraum:         2019-01-02 – 2025-12-30
Sektoren:         10 verschiedene Sektoren
Zielvariable:     Forward_Return_1M
  Mittelwert:     0.0103
  Std.-Abw.:      0.1083
  Skewness:       0.315
  Kurtosis:       5.158
NaN-Quote:        2.2% (strukturell bedingt)
Vollständige Zeilen: 126,991 (84.4%)

Erzeugte Abbildungen:
  01_forward_return_distribution.png
  02_features_distribution.png
  03_sector_boxplots.png
  04_correlation_heatmap.png
  05_feature_target_correlation.png
  06_time_series_price_volatility.png
  07_missing_values.png
  08_outlier_analysis.png
  09_pairplot.png


## Limitationen (für Abschlussarbeit)

Die folgenden Daten- und Methoden-Limitationen werden im Abschlussbericht transparent gemacht:

### L1 — Preiswahl: `Close` statt `Adj_Close`
yfinance liefert für `Adj_Close` bei Dividenden- und Split-Ereignissen teils fehlerhafte NaN-Werte. Wir nutzen daher `Close` als Preisreferenz. Dies bedeutet, dass **Dividenden und Aktiensplits nicht adjustiert** werden. Für den Zeithorizont 2019–2025 ist der Fehler gering (DAX/MDAX-Unternehmen haben in der Regel moderate Dividenden), aber nicht null.

### L2 — Survivorship Bias
Der Datensatz enthält nur Unternehmen, die im Erhebungszeitraum im DAX40 oder MDAX gelistet sind. Unternehmen, die delisted wurden oder abgestiegen sind, fehlen. Dies führt zu einer systematischen Überschätzung positiver Renditen.

### L3 — Stationarität
Kurspreise (`Close`) sind nicht stationär. Das Modell nutzt Renditen und normierte Features, die weitgehend stationär sind. Eine formale ADF-Test-Analyse wird in Notebook 13 (Feature Engineering) ergänzt.

### L4 — Look-Ahead-Bias
Die Zielvariable `Forward_Return_1M` ist korrekterweise als zukünftige Rendite definiert. Alle Features werden ausschließlich aus vergangenen Daten berechnet. Ein Look-Ahead-Bias ist durch die Datenpipeline ausgeschlossen.

### L5 — Marktregime-Wechsel
Der Zeitraum 2019–2025 umfasst sehr unterschiedliche Marktregimes (Pre-COVID-Bullenmarkt, COVID-Crash, Erholung, Zinswende, KI-Euphorie). Ein auf historischen Daten trainiertes Modell kann künftige Regime-Wechsel nicht antizipieren.

### L6 — Transaktionskosten
Das Modell ignoriert Transaktionskosten, Spread und Marktliquidität. In der Praxis würden diese die Rendite des Basket-Ansatzes erheblich reduzieren.

---
*Diese EDA-Ergebnisse fließen direkt in Notebook 13 (Feature Engineering) und Notebook 14 (Modellierung) ein.*